# Week 6 Homework: Distillation as Recipe Design

**ECBS5200 — Practical Deep Learning Engineering**

Closing homework of the term. **Budget ~6–7 hours** including memo.
Roughly: Part 1 (teacher diagnosis) ~45 min, Part 2 (recipe shortlist
+ grid selection + eval) ~2 h, Part 3 (two literature tests) ~45 min,
Part 4 (memo, 5 sections with cross-week integration and reference
tables) ~2.5–3 h. Start the memo early; the synthesis takes time.

## The frame

Distillation is one of several ways to transfer model properties.
Different recipes transfer different things at different costs. The
applied ML skill is **naming the property you need, picking the
cheapest recipe that delivers it, and knowing when no cheap
substitute exists.**

Specifically: post-hoc temperature scaling reproduces top-1
calibration almost for free — but it can't reshape the per-class
probability distribution the way distillation can. So "should I use
KD?" is the wrong question. The right question is "what property does
my deployment depend on, and what's the cheapest recipe that delivers
it?"

## Five parts

- **Part 0 (~5 min):** pick a deployment scenario + accept the data budget
- **Part 1 (~45 min):** diagnose what the teacher has worth transferring
- **Part 2 (~2h):** rank properties for your scenario, build a recipe shortlist
- **Part 3 (~45 min):** test two mechanism claims from the literature
- **Part 4 (~1.5h):** choose a recipe, defend, write the memo

No GPU training required. Analysis-only on pre-computed artifacts.
Run on Kaggle CPU or your laptop — your call.

## Setup

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → CPU is fine (this homework is analysis-only)
4. **Secrets** → add `HF_TOKEN` (read token sufficient — no uploads)

In [ ]:
# GUIDED: env config. Run as-is.
import os, sys, subprocess

if os.path.exists("/kaggle/working"):
    os.environ["HF_HOME"] = "/kaggle/working/.hf_cache"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "scikit-learn", "matplotlib", "pandas",
])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
               check=False)
print("Packages installed.")

In [ ]:
# GUIDED: HuggingFace auth. Run as-is.
from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: no HF_TOKEN found — public reads still work but rate limits hit fast.")

In [ ]:
# GUIDED: imports + helpers. Run as-is.
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def softmax(logits):
    s = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(s); return e / e.sum(axis=-1, keepdims=True)

def expected_calibration_error(probs, labels, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    confs = probs.max(axis=-1); preds = probs.argmax(axis=-1)
    correct = (preds == labels); n = len(labels); ece = 0.0
    for i in range(n_bins):
        m = (confs > edges[i]) & (confs <= edges[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / n) * abs(correct[m].mean() - confs[m].mean())
    return float(ece)

def nll_fn(probs, labels):
    return float(-np.log(probs[np.arange(len(labels)), labels].clip(min=1e-12)).mean())

def js_divergence(p, q, eps=1e-12):
    """Jensen-Shannon divergence between two probability distributions
    (per-row if 2D). Returns scalar (mean over rows if 2D).
    Symmetric, bounded in [0, ln(2)]."""
    p = np.clip(p, eps, 1.0); q = np.clip(q, eps, 1.0)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * (np.log(p) - np.log(m)), axis=-1)
    kl_qm = np.sum(q * (np.log(q) - np.log(m)), axis=-1)
    return float(np.mean(0.5 * (kl_pm + kl_qm)))

print("Imports + helpers OK.")

In [ ]:
# GUIDED: load all the artifacts you'll need across the homework.
# Vanilla and distilled students from the lab; teacher logits from
# Week 6 dataset repo; six grid configs from the homework grid.

# Lab artifacts (vanilla + distilled student val predictions)
VANILLA_REPO   = "earino/ecbs5200-week6-vanilla-baseline"
DISTILLED_REPO = "earino/ecbs5200-week6-distilled-student"

v_path = hf_hub_download(repo_id=VANILLA_REPO, repo_type="model", filename="val_predictions.npz")
d_path = hf_hub_download(repo_id=DISTILLED_REPO, repo_type="model", filename="val_predictions.npz")
v_npz = np.load(v_path); d_npz = np.load(d_path)

v_logits = v_npz["logits"].astype(np.float32); v_preds = v_npz["preds"]
v_labels = v_npz["labels"]; val_tiers = v_npz["val_tiers"]
d_logits = d_npz["logits"].astype(np.float32); d_preds = d_npz["preds"]
assert (d_npz["labels"] == v_labels).all() and (d_npz["val_tiers"] == val_tiers).all()

# Teacher val logits (from the public dataset repo built for the lab)
TEACHER_REPO = "earino/ecbs5200-week6-teacher-logits"
t_path = hf_hub_download(repo_id=TEACHER_REPO, repo_type="dataset",
                         filename="val_logits_canonical_final.npz")
t_npz = np.load(t_path)
t_logits = t_npz["logits"].astype(np.float32); t_labels = t_npz["labels"]
assert (t_labels == v_labels).all(), "teacher and student val orderings differ"

# Tier label sets
HEAD_LABELS = sorted(set(v_labels[val_tiers == "head"].tolist()))
MID_LABELS  = sorted(set(v_labels[val_tiers == "mid"].tolist()))
TAIL_LABELS = sorted(set(v_labels[val_tiers == "tail"].tolist()))
NUM_LABELS = 113

# Grid (6 configs)
GRID_REPO = "earino/ecbs5200-week6-grid-results"
GRID_CONFIGS = [(1.0, 0.7), (4.0, 0.7), (8.0, 0.7), (1.0, 0.9), (4.0, 0.9), (8.0, 0.9)]
def grid_filename(T_d, alpha):
    return f"grid_Td{int(T_d)}_a{int(round(alpha * 100))}_val_predictions.npz"

grid_npz = {}
for T_d, alpha in GRID_CONFIGS:
    p = hf_hub_download(repo_id=GRID_REPO, repo_type="dataset",
                        filename=grid_filename(T_d, alpha))
    grid_npz[(T_d, alpha)] = np.load(p)
print(f"All artifacts loaded.  vanilla/distilled/teacher + {len(grid_npz)} grid configs.")
print(f"|head|={len(HEAD_LABELS)}  |mid|={len(MID_LABELS)}  |tail|={len(TAIL_LABELS)}")

---

# Part 0 — Pick your deployment scenario (~5 min)

You will defend ONE deployment recommendation. Different deployments
care about different properties. Pick one of the three scenarios
below and commit to it before doing the analysis. **Don't switch
scenarios mid-homework** — the rest of the analysis is anchored to
the property priorities your scenario fixes.

| Scenario | Hard constraint | Primary metric | Secondary metric |
|---|---|---|---|
| **A — High-throughput batch triage** | <10 ms/example at batch=32 on T4; serve at 10k QPS | Macro F1 | Per-example cost |
| **B — Regulated escalation review** | Human reviewer sees model's top-1 confidence to decide whether to escalate; calibrated probabilities required | ECE (target ≤0.05) | NLL |
| **C — Long-tail rare-class monitoring** | Tail-class detection matters; false-negatives on tail are expensive | Tail F1 | Tail calibration |

**Note on Scenario C — read before picking it.** Scenario C is
intentionally hard. The data ceiling we've measured all term may
mean **no recipe within the data budget cleanly wins** Scenario C.
That is a legitimate, fully-credited answer: a defense that says
"the binding constraint is data, not recipe; here are the numbers
that show why; here is the experiment that would change the answer"
scores as well as any "recipe X wins" defense for the other
scenarios. Don't avoid C because it's harder to "solve"; pick C if
the problem is interesting to you and defend the constraint.

**Data budget constraint.** Your scenario has a fixed labeled
dataset — the 79,278 train+test examples already used for both the
vanilla and distilled students. You may NOT recommend "get more
data." The lab found that data shift contributed ~+0.055 macro F1
while distillation contributed ~+0.015 — a 3.6× ratio. In industry,
getting more high-quality labels is often impossible (annotation
cost, regulatory constraints, rare events). Design within what you
have.

**YOUR CHOICE:** A / B / C: ___

**YOUR REASONING (one sentence — why this scenario):**

**YOUR INFORMAL PRIOR (before doing any analysis): which recipe do
you think will win for your scenario?** (Don't be precise; just write
the gut feeling. Predict-then-observe across the whole homework — we
come back to this in Part 4.)

---

# Part 1 — Diagnose the teacher (~45 min)

Goal: name **3 or more properties** the teacher has that are
potentially worth transferring to the student. **Unranked.** Ranking
happens in Part 2 once you've committed to a scenario's priorities.

For each property, you need:

1. A **name** (what is the property?)
2. A **measurement** (how do you know the teacher has it?)
3. A **transferability assessment** (is this plausibly transferable to a
   smaller student, or is it blocked by something — e.g., the data
   ceiling on tail classes)?

### 1a — Compute the teacher's diagnostic metrics

In [ ]:
# INTERACTIVE: per-class F1 for both teacher and the two students.
# Length-113 numpy arrays.
t_per_class_f1 = ___  # FILL IN: f1_score(... average=None, labels=range(NUM_LABELS), zero_division=0)
v_per_class_f1 = ___  # FILL IN
d_per_class_f1 = ___  # FILL IN

# Distilled - vanilla per-class delta (you'll need this in 1b and Part 2)
per_class_delta = d_per_class_f1 - v_per_class_f1

# Teacher per-tier F1 + per-tier ECE
t_preds = t_logits.argmax(axis=-1)
t_probs = softmax(t_logits)
print(f"Teacher per-tier diagnostics:")
for tier_name, tier_labels in [("head", HEAD_LABELS), ("mid", MID_LABELS), ("tail", TAIL_LABELS)]:
    m = val_tiers == tier_name
    f1 = f1_score(t_labels[m], t_preds[m], labels=tier_labels, average="macro", zero_division=0)
    ece = expected_calibration_error(t_probs[m], t_labels[m])
    nll = nll_fn(t_probs[m], t_labels[m])
    print(f"  {tier_name:5s}  F1 {f1:.4f}  ECE {ece:.4f}  NLL {nll:.4f}  (n={m.sum()})")

### 1b — Inspect the teacher's confidence shape

Some properties of the teacher only show up in the *full distribution*,
not in the argmax. For 5 random val examples drawn from each tier,
print the teacher's top-3 classes and their probabilities. Look at
the *relative magnitudes*, not just whether the top class is right.

In [ ]:
# GUIDED: print teacher top-3 distributions for sample examples per tier.
rng = np.random.RandomState(SEED)
for tier_name in ["head", "mid", "tail"]:
    idxs = np.where(val_tiers == tier_name)[0]
    sample = rng.choice(idxs, size=min(5, len(idxs)), replace=False)
    print(f"\n{tier_name.upper()} examples — teacher top-3:")
    for i in sample:
        true_class = int(t_labels[i])
        top3 = np.argsort(-t_probs[i])[:3]
        s = ", ".join(f"c{int(c)}={t_probs[i, c]:.3f}" for c in top3)
        correct = "✓" if int(t_preds[i]) == true_class else "✗"
        print(f"  idx={i:5d}  true=c{true_class:3d}  {correct}  top3: {s}")

### 1c — Build the transferable properties spec

Fill in **3–5 properties** the teacher has. Use what you computed
above. Don't rank yet — Part 2 ranks against your scenario. Each row
should be one property.

**Example row (don't copy verbatim — write your own):**

- **Property:** "calibrated top-1 confidence"
- **Measurement:** teacher ECE 0.021 on val (post-temperature-scaling)
- **Transferable?** YES, plausibly via KD (transfers shape) OR via post-hoc temperature scaling on the student (~free)

**YOUR PROPERTIES (write at least 3):**

1. **Property:**
   - **Measurement:**
   - **Transferable?**

2. **Property:**
   - **Measurement:**
   - **Transferable?**

3. **Property:**
   - **Measurement:**
   - **Transferable?**

(Optional 4–5):

---

# Part 2 — Rank properties + build recipe shortlist (~2h)

Now that you've diagnosed the teacher and committed to a scenario,
rank the properties from Part 1 by deployment value FOR YOUR
SCENARIO, then design a candidate recipe shortlist.

### 2a — Rank properties for your scenario

Take the 3+ properties from Part 1. Order them by deployment value
for the scenario you picked in Part 0.

**YOUR RANKING (1 = most important for your scenario):**

1. **Property #1:** ___, **why this rank:**
2. **Property #2:** ___, **why this rank:**
3. **Property #3:** ___, **why this rank:**

Note: the same property list can rank very differently across
scenarios. Scenario A cares about throughput-friendly accuracy;
Scenario B cares about top-1 calibration; Scenario C cares about tail
behavior. The ranking IS the design choice.

### 2b — Build the recipe shortlist

Five recipes. Four are pre-computed; the fifth (wildcard) is a
config you propose but don't run. For each recipe, fill in the
table below.

**The five recipes:**

1. **Vanilla** — plain CE, no KD, no post-hoc fix. The Week 1
   baseline at full data scale.
2. **Vanilla + post-hoc temperature scaling** — fit a single T on
   a calibration fold, apply at inference. Cheap.
3. **Distilled (lab default, T_d=4, α=0.7)** — the lab's distilled
   student.
4. **One tuned grid config** — pick from the 6-config grid based on
   your scenario's primary metric. Justify the pick.
5. **Wildcard** — propose ONE recipe NOT in the grid (e.g.,
   `T_d=16`, `α=0.5`, distilled + temperature scaling, vanilla +
   label smoothing, anything from the readings). Predict from first
   principles where it would land. **Do not run it.** Justify the
   prediction.

**Example of a strong wildcard answer** (worked example for
calibration — pick a different one for your shortlist):

> *Wildcard: distilled + post-hoc temperature scaling.*
> Prediction: ECE drops below `vanilla + T` because KD fixes
> distributional shape (non-top-1 mass) while T fixes top-1
> sharpness — the two recipes correct different defects so they
> stack. Macro F1 stays roughly equal to distilled alone because
> T-scaling is argmax-invariant. **Failure mode:** if KD already
> over-flattens the distribution, an additional T > 1 over-flattens
> further and NLL gets worse. **What would flip the decision:** if
> Scenario A (throughput) — the extra inference step from T-scaling
> is negligible, ship the stacked recipe; if Scenario B (regulated)
> — the extra calibration check earns its keep; if Scenario C
> (tail) — no, the binding constraint is data, not calibration.

**Cost note for the table:** vanilla and distilled students share
the same architecture (ModernBERT-base, 149M). **Serving cost is
identical** — same forward pass, same latency, same memory. Cost
differences live in *training compute* (KD adds the precomputed
logit load + KL term computation during training, not inference).

**Recipe table (fill in for YOUR scenario's primary metric):**

| # | Recipe | Cost (train + serve) | Predicted property delivered | Measured value of primary metric | Ship for your scenario? |
|---|---|---|---|---|---|
| 1 | Vanilla | | | | |
| 2 | Vanilla + temp | | | | |
| 3 | Distilled (4, 0.7) | | | | |
| 4 | Grid config: (T_d=___, α=___) | | | | |
| 5 | Wildcard: ___ | n/a (didn't run) | | n/a (didn't run) | |

### 2c — Verify recipes 1–4 with measurements

Compute the metrics for your scenario across the four runnable
recipes. Use the helpers; the cells below are GUIDED.

**Important — read before you read the table:** All four arms in
this section are evaluated on the **eval half of val (n=3,215)**, not
the full val set the lab used (n=6,430). This is necessary because
temperature scaling needs a held-out fit set — we fit T on the cal
half (n=3,215) and evaluate all four arms on the eval half so the
comparison is fair.

Consequence: your numbers here will differ from the lab's full-val
numbers by ~1–2 points on macro F1. That is expected and is not a
bug. The sanity check below confirms your loaded artifacts match
the lab's full-val numbers before we move to the eval-fold table.

In [ ]:
# GUIDED: fit T on a 50/50 cal/eval split for vanilla.
n_val = len(v_labels)
rng_fold = np.random.RandomState(SEED + 999)
perm = rng_fold.permutation(n_val)
cal_idx = perm[: n_val // 2]
eval_idx = perm[n_val // 2 :]

def fit_temperature(logits, labels, max_iter=200):
    """LBFGS optimization of scalar T > 0 to minimize NLL.

    Parameterizes log_T so T = exp(log_T) is structurally positive — avoids
    the rare LBFGS pathology where T drifts non-positive on ill-behaved
    logits. Same pattern as Week 5's temperature_scale helper.
    """
    lt = torch.from_numpy(logits).float(); lab = torch.from_numpy(labels).long()
    log_T = torch.nn.Parameter(torch.zeros(1))  # log_T = 0 → T = 1.0 init
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=max_iter)
    nll_loss = torch.nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        T = torch.exp(log_T)
        l = nll_loss(lt / T, lab); l.backward(); return l
    opt.step(closure)
    return float(torch.exp(log_T.detach()).item())

T_vanilla = fit_temperature(v_logits[cal_idx], v_labels[cal_idx])
print(f"Fitted T for vanilla student: {T_vanilla:.4f}")

# Sanity check: vanilla + distilled macro F1 on the FULL val set
# (should match the lab's 0.2638 / 0.2789 numbers to four decimals).
# If these don't match, your loaded artifacts are not the lab's.
_full_v_f1 = f1_score(v_labels, v_preds, average="macro", zero_division=0)
_full_d_f1 = f1_score(v_labels, d_preds, average="macro", zero_division=0)
print(f"Full-val sanity check — vanilla F1 {_full_v_f1:.4f} | "
      f"distilled F1 {_full_d_f1:.4f}")
print("(These should match the lab's full-val numbers; the table below "
      "uses the eval fold only, so its F1s will differ by 1–2 pts.)")

In [ ]:
# INTERACTIVE: pick your grid config. Replace the (T_d, alpha) below
# with your scenario-justified choice from the 6-config grid.
my_grid_config = (___, ___)  # FILL IN: e.g., (1.0, 0.7) or (8.0, 0.9)
assert my_grid_config in GRID_CONFIGS, f"{my_grid_config} not in grid"
g_npz = grid_npz[my_grid_config]
g_logits = g_npz["logits"].astype(np.float32); g_preds = g_npz["preds"]

In [ ]:
# GUIDED: compute the four runnable recipes' metrics on the eval fold.
# All four arms evaluated on the SAME eval fold for a fair comparison.
def eval_arm(logits_eval, labels_eval, T=1.0):
    probs = softmax(logits_eval / T)
    preds = probs.argmax(axis=-1)
    return {
        "macro_f1": float(f1_score(labels_eval, preds, average="macro", zero_division=0)),
        "ece": expected_calibration_error(probs, labels_eval),
        "nll": nll_fn(probs, labels_eval),
        "tail_f1": float(f1_score(
            labels_eval[val_tiers[eval_idx] == "tail"],
            preds[val_tiers[eval_idx] == "tail"],
            labels=TAIL_LABELS, average="macro", zero_division=0,
        )),
    }

vanilla_raw    = eval_arm(v_logits[eval_idx], v_labels[eval_idx])
vanilla_scaled = eval_arm(v_logits[eval_idx], v_labels[eval_idx], T=T_vanilla)
distilled_lab  = eval_arm(d_logits[eval_idx], v_labels[eval_idx])
grid_picked    = eval_arm(g_logits[eval_idx], v_labels[eval_idx])

print(f"{'Arm':<28}{'F1':>8}{'ECE':>10}{'NLL':>10}{'Tail F1':>10}")
print("-" * 66)
for name, m in [("Vanilla raw",          vanilla_raw),
                ("Vanilla + temp scaling", vanilla_scaled),
                ("Distilled (4, 0.7)",   distilled_lab),
                (f"Grid config {my_grid_config}", grid_picked)]:
    print(f"{name:<28}{m['macro_f1']:>8.4f}{m['ece']:>10.4f}"
          f"{m['nll']:>10.4f}{m['tail_f1']:>10.4f}")

### 2d — Wildcard prediction (no measurement)

Propose your wildcard config. Justify the prediction from first
principles — what mechanism leads you to expect that result?

**YOUR WILDCARD:**

- **Recipe name:**
- **Why this recipe is interesting for your scenario:**
- **Predicted result vs the four measured recipes:** (better/worse on
  which metrics, and by roughly how much?)
- **Mechanism for the prediction:** (1–2 sentences)

Example (don't copy verbatim — write your own):

> **Wildcard: distilled + post-hoc temperature scaling.** I expect
> this to match the lab default's NLL (~1.33) but reduce ECE further,
> because the distilled student has the right distribution shape but
> may still be slightly miscalibrated on top-1 confidence. Adding T
> on top is a free further calibration. I don't predict an F1 change.
> Mechanism: T-scaling rescales the softmax sharpness without
> changing argmax or relative class-rank — so KD's distributional
> transfer is preserved, and the residual top-1 miscalibration gets
> fixed.

### 2e — Reconcile your shortlist

Now look at your filled-in recipe table. **In writing**, answer:

1. **Which recipe ships for your scenario?**
2. **Why?** (Reference at least one specific number from your table.)
3. **Was your Part 0 informal prior right?** If not, what was your
   wrong assumption?

YOUR ANSWERS:

---

# Part 3 — Test mechanism claims from the literature (~45 min)

Two specific claims, two specific tests. Each test is computable
from the artifacts you've already loaded.

### Test A — Stanton 2021: KD doesn't fully transfer the teacher's predictive distribution

**The claim** (Stanton et al. 2021, *Does Knowledge Distillation
Really Work?*, NeurIPS): even when KD improves the student's
generalization, the student does NOT fully recover the teacher's
predictive distribution. The mechanism is optimization difficulty,
not identifiability — the student *could* in principle match the
teacher, but optimization doesn't get there.

**The test:** measure the **Jensen-Shannon divergence** between the
teacher's softmax distribution and the student's softmax distribution,
averaged per tier. JS divergence is a symmetric, bounded distance
between probability distributions in `[0, ln 2] ≈ [0, 0.693]`.
Anchors: **JS = 0** means identical distributions; **JS ≈ 0.69** means
completely different (no overlap). Typical values for this task land
in the 0.05–0.25 range — read the *reduction* (`js_v − js_d`) as your
key number, not the absolute level. Compute for vanilla student vs
teacher, and distilled student vs teacher.

**The hypothesis:** distilled-vs-teacher JS should be SMALLER than
vanilla-vs-teacher JS (KD reduces the gap), but a substantial residual
remains, especially in the tail. KD reduces divergence; it doesn't
eliminate it.

Why this isn't argmax agreement: on a 113-class long-tail dataset,
both students will agree on the teacher's argmax for many head
examples by default. JS measures the *shape* of the distribution —
what Stanton actually argued the student fails to recover.

In [ ]:
# GUIDED: compute teacher / vanilla / distilled softmax probabilities.
v_probs = softmax(v_logits)
d_probs = softmax(d_logits)
# t_probs already computed in Part 1

In [ ]:
# INTERACTIVE: per-tier JS divergence between teacher and each student.
# Use the js_divergence helper (defined in setup).
print(f"{'Tier':<8}{'JS(teacher || vanilla)':>26}{'JS(teacher || distilled)':>28}{'reduction':>14}")
print("-" * 76)
for tier_name in ["head", "mid", "tail"]:
    m = val_tiers == tier_name
    js_v = ___  # FILL IN: js_divergence(t_probs[m], v_probs[m])
    js_d = ___  # FILL IN: js_divergence(t_probs[m], d_probs[m])
    reduction = js_v - js_d
    print(f"{tier_name:<8}{js_v:>26.4f}{js_d:>28.4f}{reduction:>+14.4f}")

### Test A — Reconcile

Read your JS table:

1. **Did distillation reduce the JS divergence vs vanilla?** Per tier?
2. **Is there a residual?** What's the magnitude?
3. **Does the reduction depend on tier?** Is the dichotomy visible
   here (head vs tail)?
4. **Busbridge connection (no measurement needed):** the slides cited
   Busbridge et al. 2025 §E.8 — full-distribution KD vs top-1-only KD
   produces a 50-100× ECE difference at LLM scale. We did
   full-distribution KD here. If a hypothetical "hard-label student"
   had been trained on the teacher's argmax outputs only (no soft
   targets), what would you predict for its JS divergence to the
   teacher, *relative to* both our students? Reason from the
   mechanism — no need to run anything.

YOUR ANSWERS:

<details>
<summary><b>Click after answering</b></summary>

What we're looking for is your articulation, not a specific result.
Stanton's claim survives a wide range of empirical patterns —
distillation typically reduces but doesn't close the teacher-student
gap. If your data shows a large reduction with a small residual, KD
is doing more than Stanton's worst-case. If the reduction is small,
KD is doing less than naive expectations would suggest. Either way,
the *residual* is the load-bearing observation: a student trained on
soft targets is not a faithful copy of the teacher.

(Optional secondary measure: argmax agreement per tier. Both students
vs the teacher. This is a coarser proxy for the same claim — JS is
the principled measure, agreement is the easy-to-explain measure.)

</details>

### Test B — Han Guo 2021: KD acts as a calibration regularizer in NLP

**The claim** (Han Guo et al. 2021, *Uncertainty Calibration for
Text Classification and the Role of Distillation*, RepL4NLP): KD's
calibration improvement reproduces a substantial fraction of what
you can get with post-hoc temperature scaling. They argue ~111% of
T-scaling's calibration win is reproducible by distillation alone.

**The test:** quantify the fraction of the distilled student's ECE
improvement that is reproducible by post-hoc temperature scaling on
the vanilla student. Three arms on the same eval fold:

- Arm 1: **Vanilla raw** (the baseline)
- Arm 2: **Vanilla + post-hoc T** (the cheap fix)
- Arm 3: **Distilled** (the expensive recipe)

Compute the three ECEs and the fraction:

`gap_v_to_vT = ECE_vanilla − ECE_vanilla_T` (post-hoc fix's gain)
`gap_v_to_d  = ECE_vanilla − ECE_distilled` (KD's gain)
`fraction_reproducible = gap_v_to_vT / gap_v_to_d`

Han Guo argues this fraction is large (close to 1, sometimes >1).
What do you find on this dataset?

In [ ]:
# INTERACTIVE: compute the three-arm ECE comparison (you have the
# arms already — vanilla_raw, vanilla_scaled, distilled_lab from
# Part 2). Just lift the ECE values and compute the fraction.

ece_v   = ___  # FILL IN: vanilla_raw["ece"]
ece_vT  = ___  # FILL IN: vanilla_scaled["ece"]
ece_d   = ___  # FILL IN: distilled_lab["ece"]

gap_v_to_vT = ece_v - ece_vT
gap_v_to_d  = ece_v - ece_d
fraction = gap_v_to_vT / max(gap_v_to_d, 1e-9)

print(f"{'Arm':<28}{'ECE':>10}{'NLL':>10}{'Macro F1':>12}")
print("-" * 60)
print(f"{'Vanilla raw':<28}{vanilla_raw['ece']:>10.4f}{vanilla_raw['nll']:>10.4f}{vanilla_raw['macro_f1']:>12.4f}")
print(f"{'Vanilla + temp scaling':<28}{vanilla_scaled['ece']:>10.4f}{vanilla_scaled['nll']:>10.4f}{vanilla_scaled['macro_f1']:>12.4f}")
print(f"{'Distilled (4, 0.7)':<28}{distilled_lab['ece']:>10.4f}{distilled_lab['nll']:>10.4f}{distilled_lab['macro_f1']:>12.4f}")
print()
print(f"Δ ECE (vanilla → vanilla+T):    {gap_v_to_vT:+.4f}  (post-hoc fix's gain)")
print(f"Δ ECE (vanilla → distilled):    {gap_v_to_d:+.4f}  (KD's gain)")
print(f"Fraction reproducible by T:     {fraction*100:.0f}%")

### Test B — Reconcile (read carefully — this is metric-specific)

Look at the three-arm table. Then answer:

1. **What fraction of KD's ECE gain is reproducible by post-hoc T?**
   (Han Guo argues "most of it" — does that hold here?)
2. **Now look at NLL.** Does temperature scaling close the same
   fraction of the NLL gap? Or does the picture flip?
3. **Why might ECE and NLL behave differently?** Hint: ECE measures
   only top-1 confidence; NLL is sensitive to the entire 113-dim
   probability distribution. Temperature scaling is a uniform
   rescaling — what does that mean for what it can and can't fix?

YOUR ANSWERS:

<details>
<summary><b>Click after answering</b></summary>

This dataset shows a striking pattern: post-hoc temperature scaling
can produce *better ECE than distillation* (~0.025 vs ~0.055), but
*worse NLL* (~1.43 vs ~1.33). The fraction reproducible on ECE
may exceed 100% — temperature scaling overshoots distillation on
top-1 calibration.

**Why metric-specific?** ECE only sees the *top-1* confidence's
miscalibration. Temperature scaling is *exactly* the right tool for
that: one parameter rescales the sharpness of the softmax. NLL is
sensitive to the entire 113-dim distribution; uniform rescaling
can't change *relative* probabilities of non-top-1 classes. KD
transfers that relative structure (the 12% chance class 53, 7%
chance class 89), which post-hoc T cannot reproduce.

**Implication for your deployment defense (Part 4):** match metric to
scenario. If your scenario uses top-1 confidence thresholds (e.g.,
Scenario B's escalation routing), ECE matters and temperature scaling
may dominate. If your scenario consumes the full distribution
(ensembling, downstream Bayesian inference, second-class human
review), NLL matters and KD wins.

</details>

---

# Part 4 — Memo (~1.5h)

Five prompts. The rubric is in `assessments/week6_memo_rubric.md`.
Aim for 2–3 pages, HTML upload to Moodle by Wednesday morning before
next class. Tables and figures don't count toward the page limit.

**Reference-grade evidence is in the cells you already ran.** Don't
re-do anything; cite the numbers you computed.

## §3c carry-overs from the lab

Use the coverage_table you computed in the lab to fill these in. If
you didn't save it, the cell below recomputes.

In [ ]:
# GUIDED: re-build the coverage table. `n_covered` is reported alongside
# coverage and accuracy because at high thresholds the covered count can
# drop into the dozens — accuracy 1.0 on 8 examples is descriptive, not
# a population estimate. Always cite n alongside.
def coverage_table(probs, labels, thresholds=(0.5, 0.6, 0.7, 0.8, 0.9)):
    confs = probs.max(axis=-1); preds = probs.argmax(axis=-1); rows = []
    for t in thresholds:
        cov_mask = confs >= t
        n_covered = int(cov_mask.sum())
        coverage = float(cov_mask.mean())
        accuracy = float((preds[cov_mask] == labels[cov_mask]).mean()) if n_covered > 0 else float("nan")
        rows.append((t, coverage, accuracy, n_covered))
    return rows

v_cov = coverage_table(softmax(v_logits), v_labels)
d_cov = coverage_table(softmax(d_logits), v_labels)
print(f"{'Thresh':<8}{'Van.cov':>9}{'Van.acc':>9}{'Van.n':>8}    "
      f"{'Dist.cov':>10}{'Dist.acc':>10}{'Dist.n':>8}")
print("-" * 72)
for (t, cv, av, nv), (_, cd, ad, nd) in zip(v_cov, d_cov):
    print(f"{t:<8.2f}{cv:>9.3f}{av:>9.3f}{nv:>8d}    "
          f"{cd:>10.3f}{ad:>10.3f}{nd:>8d}")

### Q1. Vanilla coverage @ 0.70 vs distilled coverage @ 0.70
- Vanilla coverage @ 0.70: ___, accuracy: ___, n_covered: ___
- Distilled coverage @ 0.70: ___, accuracy: ___, n_covered: ___

### Q2. SLA: auto-routed examples must be ≥ 95% accurate
Report the **lowest** threshold from {0.5, 0.6, 0.7, 0.8, 0.9} where
accuracy ≥ 0.95. If no threshold in the table reaches 95% accuracy,
answer "none" and report the highest accuracy you observed.

- Vanilla: threshold ___, coverage at that threshold ___, n_covered ___
- Distilled: threshold ___, coverage at that threshold ___, n_covered ___

### Q3. Operational summary (one sentence)
What is the operational difference between a well-calibrated and a
poorly-calibrated model when the downstream system uses confidence
thresholds?

YOUR ANSWER:

## Memo prompts (~half a page each, with tables/figures liberally)

Rubric weights: 25 / 15 / 15 / 15 / 30. Prompt 5 is the capstone and
seeds your Final Model Decision Dossier (separate 15% deliverable).

### Prompt 1 — What did distillation transfer? (25 pts)

Per-tier F1 deltas (distilled − vanilla) and per-tier ECE deltas from
the lab. Then: integrate Part 1 (your transferable properties spec),
Part 3 Test A (JS divergence per tier), and Part 3 Test B (the
metric-specific calibration finding).

**The framing the rubric rewards:** distillation transfers
*distributional structure* — the relative probabilities across all
113 classes. That property is **measurable in NLL but not in ECE**,
and **measurable in JS divergence but not in argmax agreement**. Show
this with your numbers.

### Prompt 2 — What didn't distillation fix? (15 pts)

The data ceiling — tail F1 stays low for student, vanilla, AND
teacher. Defend: is this a property of the task, the data, or the
method? What additional measurements would you make to tell?

Required evidence:
- Cross-model tail F1: vanilla / distilled / teacher
- The data-confound finding (~+0.055 from data shift vs +0.015 from KD)
- At least one specific additional measurement that would falsify the
  data-ceiling hypothesis

### Prompt 3 — Hyperparameters and the noise floor (15 pts)

Use Part 2's recipe shortlist:
- Did the lab default `(T_d=4, α=0.7)` dominate any single metric?
- For your scenario's primary metric, was the spread across recipes
  inside or outside the bootstrap CI you measured in the lab?
- Anchor your answer to a specific tier and a specific bootstrap CI.
- What did your wildcard prediction add that the grid couldn't show?

### Prompt 4 — Three compressions, three lessons (15 pts)

Label compression (Week 4–5 pipeline reveal), weight compression
(Week 5 quantization), knowledge compression (Week 6).

- Which two interact most strongly on this dataset?
- Which two are nearly orthogonal?
- Defend each claim with a number.

### Prompt 5 — Defend the deployment (30 pts)

Pick one recipe — your shortlist winner — and defend it for the
scenario you committed to in Part 0.

Required evidence:

- The scenario you chose (A / B / C) and its primary + secondary
  metrics.
- **Specific numbers** from §3c (vanilla cov @ 0.70, distilled acc @
  0.70, the 95% SLA threshold for each model).
- Per-tier F1 / ECE from the lab.
- Your Part 3 finding stated **metric-by-metric**: how does
  temperature scaling compare to distillation on (a) ECE,
  (b) NLL, (c) macro F1, (d) JS divergence to teacher? They likely
  point in different directions — the right tool depends on which
  metric your scenario consumes.
- **Match metric to scenario.** If your scenario uses top-1
  confidence thresholds, ECE is the relevant metric. If your
  downstream consumes the full probability distribution, NLL is the
  relevant metric.
- One specific constraint that would flip your answer (named with a
  threshold and an alternative recipe).
- Cross-week reach: if your deployment also involves quantization,
  what does Week 5's findings imply about the combined recipe?

**This prompt seeds the Final Model Decision Dossier** (15% of the
course grade, separate deliverable). A complete memo answer here is
the skeleton of the dossier; the dossier expands it.

---

### End of Week 6 homework

**Submit:** HTML export of this notebook with all interpretation
fields filled in, plus the memo as a separate HTML/PDF (linked from
the notebook is fine).

**Course closure:** the dossier is the only remaining deliverable
before the final exam. Save your work — Prompt 5 above is its
starting point.